# 04 — Correcting a small deletion: prime editing beyond the substitution

**AlleleForge is a research tool. It is not a medical device and provides no medical advice.**
Off-target nominations are computational and must be experimentally validated.

Most monogenic disease that prime editing exists for is not a point substitution — it is a small
indel. The canonical example is CFTR **ΔF508**, an in-frame 3-bp deletion carried by the large
majority of people with cystic fibrosis. This notebook designs the correcting pegRNA for a
ΔF508-shaped deletion and shows the one thing that makes it possible: the RT template is built at
**variable length**, so it can write back bases the genome no longer has.

The RT template always reads *5' homology (nick → edit) + the desired allele + 3' homology*. Two
consequences fall out, and both are visible below:

* a **deleted** span costs no template length — the RTT simply skips it, so a 44-bp deletion is as
  cheap to write as a 1-bp one;
* an **inserted** one costs a base each, which is why the bound that binds is on the allele the RTT
  must *write* (`PRIME_MAX_TEMPLATED_EDIT`), not on the span it replaces.

In [ ]:
import random
import tempfile
from pathlib import Path

from alleleforge.design.prime import design_prime
from alleleforge.design.routing import route
from alleleforge.genome.reference import ReferenceGenome
from alleleforge.types.edit import EditIntent
from alleleforge.types.sequence import DNASequence
from alleleforge.variant.resolver import resolve

# A small synthetic locus. Random sequence (fixed seed) is used deliberately: it
# carries real NGG PAMs on both strands, so nothing about the design is planted.
EDIT_POS = 200
REF_ALLELE, ALT_ALLELE = "ACTT", "A"  # an in-frame 3-bp deletion, ΔF508-shaped

rng = random.Random(20260909)
filler = "".join(rng.choice("ACGT") for _ in range(420))
contig = filler[:EDIT_POS] + REF_ALLELE + filler[EDIT_POS + len(REF_ALLELE) :]
fasta = Path(tempfile.mkdtemp()) / "locus.fa"
fasta.write_text(f">chr7\n{contig}\n")
reference = ReferenceGenome(fasta, build="hg38")

resolved = resolve(f"chr7:{EDIT_POS + 1}:{REF_ALLELE}>{ALT_ALLELE}", reference=reference)
print("variant:", resolved.variant, "| class:", resolved.variant.variant_class.value)

## 1. Routing admits prime — and only prime

The router is data, not a special case. A base editor installs one transition and cannot make an
indel; the nuclease is for disruption. Prime is the only chemistry that can write this edit, and
routing says so with its reasoning attached rather than returning an empty menu.

In [ ]:
for decision in route(resolved, EditIntent.CORRECT):
    verdict = "eligible" if decision.eligible else "not eligible"
    print(f"{decision.chemistry.value:<14} {verdict}")
print()
prime = next(d for d in route(resolved, EditIntent.CORRECT) if d.chemistry.value == "prime")
print("why prime:", prime.rationale)

## 2. Design the correcting pegRNA

The patient's genome carries the deletion; correcting it means **writing the three missing bases
back**. `design_prime` enumerates pegRNAs on both strands against the genome the patient actually
has, scores efficiency and outcome with calibrated uncertainty, and ranks them.

In [ ]:
candidates = design_prime(
    resolved,
    EditIntent.CORRECT,
    reference=reference,
    max_candidates=5,
    run_offtarget=False,  # keep the notebook fast; see notebook 02 for the off-target engine
)
top = candidates[0]
peg = top.pegrna

print("pegRNA spacer :", peg.spacer.sequence, "on", peg.placement.strand.value, "strand")
print("PBS / RTT     :", len(peg.pbs), "/", len(peg.rtt), "nt")
print(
    "RTT geometry  :",
    f"{peg.rtt_homology_5prime} nt 5' homology"
    f" + {peg.templated_edit_length} nt written"
    f" + {peg.rtt_homology_3prime} nt 3' homology",
)
print(
    "efficiency    :",
    round(top.efficiency.value, 3),
    "80% interval",
    tuple(round(x, 2) for x in top.efficiency.interval),
)

## 3. Read the RT template and see the edit inside it

The RTT is stored reverse-complemented (it is copied 3'→5' onto the nicked strand). Reverse
complement it and the three-part structure is literally readable: homology, the restored allele,
homology. This is the check worth making on any prime design — the reagent should *show* you the
edit it writes.

In [ ]:
template = str(DNASequence(str(peg.rtt)).reverse_complement())
lo = peg.rtt_homology_5prime
hi = lo + peg.templated_edit_length

print("RT product written at the nick (5'->3'):", template)
print("  5' homology :", template[:lo])
print("  restored    :", template[lo:hi], " <- the reference allele, written back")
print("  3' homology :", template[hi:])
assert template[lo:hi] == REF_ALLELE

## 4. The deleted span costs no template

Run the *same* variant in the opposite direction — `INSTALL`, which writes the deletion rather than
correcting it — and the RT template collapses. Installing a 3-bp deletion writes a single anchor
base, so the RTT carries only its two homology arms. That asymmetry is the whole reason a large
deletion stays designable while a large insertion does not.

In [ ]:
installing = design_prime(
    resolved, EditIntent.INSTALL, reference=reference, max_candidates=5, run_offtarget=False
)
peg_del = installing[0].pegrna

print(f"{'direction':<28}{'RTT':>5}{'written':>9}")
print(f"{'CORRECT (restore 3 bp)':<28}{len(peg.rtt):>5}{peg.templated_edit_length:>9}")
print(f"{'INSTALL (delete 3 bp)':<28}{len(peg_del.rtt):>5}{peg_del.templated_edit_length:>9}")

Swap the synthetic locus for hg38 and the real CFTR coordinates, supply a gnomAD database for the
population-aware off-target scan (notebook 02), and load trained PRIDICT2.0 weights through the
model zoo — the call shape is identical. What changes with a real reference is the *numbers*, never
the honesty: a calibrated interval on every prediction, an OOD flag when the efficiency model is
outside its training distribution, and an RT template you can read the edit out of.